# Indoor Location Prediction: Data Preprocessing & Feature Engineering Pipeline
**Project 9 · CSAI Student Project · Phase 1: Machine Learning Evaluation**

## 1. Overview and Engineering Objectives
This notebook builds the complete preprocessing and feature engineering pipeline for the UJIIndoorLoc dataset.
Following the project requirements:
1. **Missing Signal Convention**: The dataset encodes unobserved access points as `+100 dBm`. Real signals are in $[-104, 0]$ dBm. We substitute `+100` with an unobserved threshold of `-110 dBm`.
2. **Positive Normalization**: Signals are mapped to a $[0, 1]$ interval via `(RSSI - (-110)) / (0 - (-110))`. An absent signal maps strictly to `0.0`, while a strong direct line-of-sight signal approaches `1.0`.
3. **Dead WAP Pruning**: 55 WAPs that never register any observation across the entire training set are identified and dropped.
4. **Empty Scan Removal**: 76 training scans where no access points were detected are pruned to prevent label pollution.
5. **Engineered Signal Descriptors**:
   - `signal_count`: Total number of active APs detected in the fingerprint.
   - `max_signal`: Normalized RSSI of the strongest detected AP (direct indicator of proximity to nearest router).
   - `mean_signal`: Average normalized RSSI of visible APs.
   - `std_signal`: Standard deviation of visible RSSIs.
6. **No-Leakage Serialization**: Feature definitions and transformations are fitted exclusively on `trainingData.csv` and stored for downstream model training.


In [1]:
import os
import numpy as np
import pandas as pd
import joblib

print('Importing libraries and configuring paths...')
data_dir = '../Datasets/UJIndoorLoc' if os.path.exists('../Datasets/UJIndoorLoc') else 'Datasets/UJIndoorLoc'
train_path = os.path.join(data_dir, 'trainingData.csv')
val_path = os.path.join(data_dir, 'validationData.csv')

raw_train = pd.read_csv(train_path)
raw_val = pd.read_csv(val_path)

print(f'Raw train shape: {raw_train.shape}')
print(f'Raw val shape:   {raw_val.shape}')


Importing libraries and configuring paths...


Raw train shape: (19937, 529)
Raw val shape:   (1111, 529)


## 2. Invariant Audits & Pruning Empty Scans
We drop training scans that contain 0 visible access points (`WAP == 100` for all 520 WAPs).


In [2]:
wap_cols = [c for c in raw_train.columns if c.startswith('WAP')]
train_active_counts = (raw_train[wap_cols] != 100).sum(axis=1)

# Drop empty scans
train_clean = raw_train[train_active_counts > 0].copy()
val_clean = raw_val.copy()

print(f'Dropped {len(raw_train) - len(train_clean)} empty training scans.')
print(f'Retained training samples: {len(train_clean):,}')

# Identify dead WAPs in training set
active_waps = [w for w in wap_cols if (train_clean[w] != 100).any()]
dead_waps = [w for w in wap_cols if (train_clean[w] == 100).all()]

print(f'Active informative WAPs: {len(active_waps)}')
print(f'Pruned dead WAPs:        {len(dead_waps)}')


Dropped 76 empty training scans.
Retained training samples: 19,861
Active informative WAPs: 465
Pruned dead WAPs:        55


## 3. Signal Transformation and Feature Engineering Function
Defines the reproducible feature extractor.


In [3]:
def extract_signal_features(df, active_wap_list, missing_val=-110.0, max_val=0.0):
    '''
    Transforms raw RSSI values and computes high-level spatial signal descriptors.
    '''
    # 1. Replace +100 with baseline missing dBm
    raw_rssi = df[active_wap_list].replace(100, missing_val).to_numpy(dtype=np.float32)
    
    # 2. Linear scaling to [0, 1]
    denom = max_val - missing_val
    norm_rssi = np.clip((raw_rssi - missing_val) / denom, 0.0, 1.0)
    
    # 3. Visible AP boolean mask
    is_visible = (raw_rssi > missing_val)
    signal_count = is_visible.sum(axis=1, keepdims=True).astype(np.float32)
    
    # 4. Strongest observed signal
    max_signal = np.where(signal_count > 0, norm_rssi.max(axis=1, keepdims=True), 0.0)
    
    # 5. Mean observed signal
    sum_signal = norm_rssi.sum(axis=1, keepdims=True)
    mean_signal = np.where(signal_count > 0, sum_signal / np.maximum(signal_count, 1.0), 0.0)
    
    # 6. Standard deviation of observed signals
    diff_sq = np.where(is_visible, (norm_rssi - mean_signal) ** 2, 0.0)
    std_signal = np.where(signal_count > 1, np.sqrt(diff_sq.sum(axis=1, keepdims=True) / np.maximum(signal_count, 1.0)), 0.0)
    
    # Stack all features
    X = np.hstack([norm_rssi, signal_count, max_signal, mean_signal, std_signal])
    feature_names = list(active_wap_list) + ['signal_count', 'max_signal', 'mean_signal', 'std_signal']
    return X, feature_names

X_train, feature_names = extract_signal_features(train_clean, active_waps)
X_val, _ = extract_signal_features(val_clean, active_waps)

print(f'Processed X_train shape: {X_train.shape}')
print(f'Processed X_val shape:   {X_val.shape}')
print(f'Total engineered features: {len(feature_names)}')


Processed X_train shape: (19861, 469)
Processed X_val shape:   (1111, 469)
Total engineered features: 469


## 4. Extract Multi-Task Target Vectors
We construct the targets:
- `y_bldg`: Building ID (0, 1, 2)
- `y_floor`: Floor Level (0, 1, 2, 3, 4)
- `y_coords`: Spatial coordinates `[LATITUDE, LONGITUDE]`


In [4]:
# Training targets
y_train_bldg = train_clean['BUILDINGID'].to_numpy(dtype=np.int64)
y_train_floor = train_clean['FLOOR'].to_numpy(dtype=np.int64)
y_train_coords = train_clean[['LATITUDE', 'LONGITUDE']].to_numpy(dtype=np.float64)

# Validation targets
y_val_bldg = val_clean['BUILDINGID'].to_numpy(dtype=np.int64)
y_val_floor = val_clean['FLOOR'].to_numpy(dtype=np.int64)
y_val_coords = val_clean[['LATITUDE', 'LONGITUDE']].to_numpy(dtype=np.float64)

print('Target verification:')
print('Train Building classes:', np.unique(y_train_bldg, return_counts=True))
print('Train Floor classes:   ', np.unique(y_train_floor, return_counts=True))
print('Train Coords range Lat:', y_train_coords[:, 0].min(), 'to', y_train_coords[:, 0].max())
print('Train Coords range Lon:', y_train_coords[:, 1].min(), 'to', y_train_coords[:, 1].max())


Target verification:
Train Building classes: (array([0, 1, 2]), array([5248, 5159, 9454]))
Train Floor classes:    (array([0, 1, 2, 3, 4]), array([4332, 5001, 4415, 5011, 1102]))
Train Coords range Lat: 4864745.745015971 to 4865016.687799998
Train Coords range Lon: -7691.338399998844 to -7300.818990092725


## 5. Artifact Serialization
Save the preprocessed matrices and metadata for seamless execution in the modeling notebook.


In [5]:
out_dir = '../Datasets/processed' if os.path.exists('../Datasets') else 'Datasets/processed'
os.makedirs(out_dir, exist_ok=True)

processed_data = {
    'X_train': X_train,
    'X_val': X_val,
    'y_train_bldg': y_train_bldg,
    'y_val_bldg': y_val_bldg,
    'y_train_floor': y_train_floor,
    'y_val_floor': y_val_floor,
    'y_train_coords': y_train_coords,
    'y_val_coords': y_val_coords,
    'active_waps': active_waps,
    'feature_names': feature_names
}

out_file = os.path.join(out_dir, 'preprocessed_wifi_data.joblib')
joblib.dump(processed_data, out_file)
print(f'Successfully saved preprocessed artifacts to: {out_file}')


Successfully saved preprocessed artifacts to: ../Datasets/processed/preprocessed_wifi_data.joblib
